# Build cycle_summary (test: 2 files)

Exports all `summary` columns for each cycle. When this looks correct, change `NUM_FILES` to `None` and run a full export to `cycle_summary.csv`.

In [ ]:
import os
import sys
import time
os.chdir('..')
import json
import pandas as pd
import glob

sys.path.insert(0, 'scripts')
from dedupe_policy import is_kept_file

NUM_FILES = None  # set to 2 for a quick test; None = all files
OUTPUT_PATH = 'data/processed/cycle_summary.csv'

def fmt_seconds(seconds):
    """Format seconds as 45.2s or 3m 12s."""
    if seconds < 60:
        return f"{seconds:.1f}s"
    minutes, secs = divmod(int(seconds), 60)
    return f"{minutes}m {secs}s"

os.makedirs('data/processed', exist_ok=True)

files = sorted(glob.glob('data/raw/FastCharge*.json'))
if NUM_FILES is not None:
    files = files[:NUM_FILES]
print(f"Processing {len(files)} file(s)")
print(f"Output: {OUTPUT_PATH}")

Processing 140 file(s)
Output: data/processed/cycle_summary.csv


In [2]:
parts = []
loop_start = time.perf_counter()

for i, file in enumerate(files):
    file_id = os.path.basename(file)
    if not is_kept_file(file_id):
        continue

    file_start = time.perf_counter()

    with open(file) as f:
        data = json.load(f)

    summary = pd.DataFrame(data['summary'])
    summary['file_id'] = file_id
    summary['cell_id'] = data['barcode']
    parts.append(summary)

    file_elapsed = time.perf_counter() - file_start
    total_elapsed = time.perf_counter() - loop_start
    done = i + 1
    avg_per_file = total_elapsed / done
    remaining = len(files) - done
    eta = avg_per_file * remaining

    print(
        f"[{done}/{len(files)}] {os.path.basename(file)} | "
        f"this file: {fmt_seconds(file_elapsed)} | "
        f"running total: {fmt_seconds(total_elapsed)} | "
        f"ETA: {fmt_seconds(eta) if remaining else '0s'}"
    )

process_elapsed = time.perf_counter() - loop_start

concat_start = time.perf_counter()
cycle_summary = pd.concat(parts, ignore_index=True)
cols = ['file_id', 'cell_id'] + [c for c in cycle_summary.columns if c not in ('file_id', 'cell_id')]
cycle_summary = cycle_summary[cols]
concat_elapsed = time.perf_counter() - concat_start

print()
print(f"Load + parse all files: {fmt_seconds(process_elapsed)}")
print(f"Concat DataFrame:       {fmt_seconds(concat_elapsed)}")
print(f"Rows: {len(cycle_summary):,}")
print(f"Columns ({len(cycle_summary.columns)}):", list(cycle_summary.columns))

[1/140] FastCharge_000000_CH19_structure.json | this file: 2.2s | running total: 2.2s | ETA: 5m 8s
[2/140] FastCharge_000001_CH16_structure.json | this file: 3.1s | running total: 5.4s | ETA: 6m 9s
[3/140] FastCharge_000001_CH30_structure.json | this file: 3.7s | running total: 9.0s | ETA: 6m 52s
[4/140] FastCharge_000001_CH38_structure.json | this file: 2.6s | running total: 11.7s | ETA: 6m 36s
[5/140] FastCharge_000002_CH10_structure.json | this file: 4.6s | running total: 16.3s | ETA: 7m 19s
[6/140] FastCharge_000002_CH18_structure.json | this file: 3.9s | running total: 20.2s | ETA: 7m 31s
[7/140] FastCharge_000002_CH26_structure.json | this file: 0.1s | running total: 20.3s | ETA: 6m 26s
[8/140] FastCharge_000002_CH2_structure.json | this file: 3.5s | running total: 23.9s | ETA: 6m 33s
[9/140] FastCharge_000002_CH34_structure.json | this file: 3.8s | running total: 27.7s | ETA: 6m 43s
[10/140] FastCharge_000002_CH42_structure.json | this file: 5.8s | running total: 33.5s | ETA: 7m

In [3]:
print("Rows per cell:")
print(cycle_summary.groupby('cell_id').size())

cycle_summary.head(10)

Rows per cell:
cell_id
EL150800453113     720
EL150800453773     618
EL150800460433     891
EL150800460436     758
EL150800460468    1055
                  ... 
el150800737390     797
el150800739476    1049
el150800739477    1094
el150800739484    1802
el150800739495     787
Length: 135, dtype: int64


,cell_id,cycle_index,discharge_capacity,charge_capacity,discharge_energy,charge_energy,dc_internal_resistance,temperature_maximum,temperature_average,temperature_minimum,date_time_iso,energy_efficiency,charge_throughput,energy_throughput,charge_duration,time_temperature_integrated,paused
0,el150800440551,0,1.934572,1.417352,6.116106,4.673017,0.029384,34.168961,30.977694,25.237902,2017-07-01T03:52:32+00:00,1.308813,1.417352,4.673017,33280.0,39217.381470,0
1,el150800440551,1,1.045426,1.045648,3.173665,3.646161,0.017864,34.850071,32.641014,30.334539,2017-07-02T00:59:44+00:00,0.870413,2.463000,8.319178,640.0,1936.503715,0
2,el150800440551,2,1.048037,1.048442,3.176155,3.651186,0.017929,34.573605,32.274544,29.808897,2017-07-02T01:59:28+00:00,0.869897,3.511442,11.970364,640.0,1915.932650,0
3,el150800440551,3,1.048029,1.047885,3.175840,3.651785,0.018012,34.393002,32.078480,30.053253,2017-07-02T02:59:12+00:00,0.869668,4.559327,15.622149,640.0,1974.178617,0
4,el150800440551,4,1.049100,1.049186,3.182562,3.653328,0.017671,34.299545,32.473103,30.322723,2017-07-02T04:01:04+00:00,0.871140,5.608512,19.275476,512.0,1930.615568,0
5,el150800440551,5,1.049680,1.049651,3.180693,3.654494,0.017613,35.859211,32.628868,29.459372,2017-07-02T05:00:48+00:00,0.870351,6.658164,22.929972,640.0,1935.573775,0
6,el150800440551,6,1.049286,1.049434,3.180875,3.654901,0.017461,34.923916,32.479107,30.657457,2017-07-02T06:00:32+00:00,0.870304,7.707598,26.584871,640.0,1927.326766,0
7,el150800440551,7,1.049450,1.049554,3.183569,3.654037,0.017321,34.685139,32.492977,30.553493,2017-07-02T07:00:16+00:00,0.871247,8.757152,30.238909,640.0,1930.954696,0
8,el150800440551,8,1.049421,1.049541,3.178062,3.653120,0.017379,34.089195,32.192825,29.450090,2017-07-02T08:00:00+00:00,0.869958,9.806692,33.892029,640.0,1980.750529,0
9,el150800440551,9,1.049059,1.049103,3.179124,3.653833,0.017379,34.245373,32.031075,30.329714,2017-07-02T09:01:52+00:00,0.870079,10.855796,37.545864,512.0,1904.153821,0


In [4]:
save_start = time.perf_counter()
cycle_summary.to_csv(OUTPUT_PATH, index=False)
save_elapsed = time.perf_counter() - save_start

print(f"Saved to {OUTPUT_PATH} ({fmt_seconds(save_elapsed)})")
print()
print("--- Timing summary ---")
print(f"Load + parse:  {fmt_seconds(process_elapsed)}")
print(f"Concat:        {fmt_seconds(concat_elapsed)}")
print(f"Write CSV:     {fmt_seconds(save_elapsed)}")
print(f"TOTAL:         {fmt_seconds(process_elapsed + concat_elapsed + save_elapsed)}")

Saved to data/processed/cycle_summary.csv (2.1s)

--- Timing summary ---
Load + parse:  8m 57s
Concat:        0.1s
Write CSV:     2.1s
TOTAL:         8m 59s
